In [14]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention
from tensorflow.keras.callbacks import EarlyStopping

In [33]:
df = pd.read_csv("/kaggle/input/datasets/preetviradiya/english-hindi-dataset/Dataset_English_Hindi.csv")

df = df.sample(70000)
df = df.dropna()

 
df['Hindi'] = df['Hindi'].astype(str)
df['Hindi'] = '<start> ' + df['Hindi'] + ' <end>'

print(df['Hindi'].head())

83777     <start> सो , अफगानिस्तान के ज्यादातर हिस्सों म...
44767     <start> मुझे लगता है जैसे, 'मुझे कुंग-फ़ू आता ...
29153                       <start> जानकारी देना|भाष् <end>
66246     <start> यद्यपि वे धर्म को अनुष्ठानों और कर्मका...
127674    <start> जाति व्यवसथा ने स्थानीय तथा ऊपर की विध...
Name: Hindi, dtype: object


In [34]:
# TOKENIZATION

 
eng_tokenizer = Tokenizer(num_words=15000, oov_token="<unk>", filters='')
hin_tokenizer = Tokenizer(num_words=15000, oov_token="<unk>", filters='')

eng_tokenizer.fit_on_texts(df['English'])
hin_tokenizer.fit_on_texts(df['Hindi'])

In [35]:
# VERIFY TOKENS

print("Start:", hin_tokenizer.word_index.get('<start>'))
print("End:", hin_tokenizer.word_index.get('<end>'))

Start: 2
End: 3


In [36]:
## SEQUENCES
max_len = 25

eng_seq = eng_tokenizer.texts_to_sequences(df['English'])
hin_seq = hin_tokenizer.texts_to_sequences(df['Hindi'])

eng_seq = pad_sequences(eng_seq, maxlen=max_len, padding='post')
hin_seq = pad_sequences(hin_seq, maxlen=max_len, padding='post')

decoder_input = hin_seq[:, :-1]
decoder_output = hin_seq[:, 1:]
decoder_output = np.expand_dims(decoder_output, -1)

In [37]:
## MODEL

vocab_eng = len(eng_tokenizer.word_index) + 1
vocab_hin = len(hin_tokenizer.word_index) + 1
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(max_len,))
enc_emb = Embedding(vocab_eng, latent_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, dropout=0.3)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

# Decoder
decoder_inputs = Input(shape=(max_len-1,))
dec_emb = Embedding(vocab_hin, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, dropout=0.3)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

# Attention
attention = Attention()
attention_out = attention([decoder_outputs, encoder_outputs])

concat = Dense(latent_dim, activation="tanh")(attention_out)

decoder_dense = Dense(vocab_hin, activation='softmax')
decoder_outputs = decoder_dense(concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

 

In [38]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 24)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 25, 256)   │ 18,619,904 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 24, 256)   │ 17,922,304 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 25, 256), │    525,312 │ embedding_6[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, 24, 256), │    525,312 │ embedding_7[0][0… │
│                     │ (None, 256),      │            │ lstm_6[0][1],     │
│                     │ (None, 256)]      │            │ lstm_6[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_3         │ (None, 24, 256)   │          0 │ lstm_7[0][0],     │
│ (Attention)         │                   │            │ lstm_6[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 24, 256)   │     65,792 │ attention_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 24, 70009) │ 17,992,313 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 55,650,937 (212.29 MB)

 Trainable params: 55,650,937 (212.29 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [44]:
model.fit(
    [eng_seq, decoder_input],
    decoder_output,
    epochs=20,
    batch_size=60,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.4104 - loss: 4.4963 - val_accuracy: 0.4930 - val_loss: 3.5965
Epoch 2/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 315s 338ms/step - accuracy: 0.4977 - loss: 3.5020 - val_accuracy: 0.5059 - val_loss: 3.3361
Epoch 3/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.5133 - loss: 3.2388 - val_accuracy: 0.5198 - val_loss: 3.1744
Epoch 4/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 315s 338ms/step - accuracy: 0.5281 - loss: 3.0316 - val_accuracy: 0.5293 - val_loss: 3.0363
Epoch 5/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.5363 - loss: 2.8641 - val_accuracy: 0.5340 - val_loss: 2.9438
Epoch 6/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.5466 - loss: 2.7032 - val_accuracy: 0.5383 - val_loss: 2.8764
Epoch 7/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.5548 - loss: 2.5749 - val_accuracy: 0.5439 - val_loss: 2.8230
Epoch 8/20
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.5639 -

In [45]:
model.fit(
    [eng_seq, decoder_input],
    decoder_output,
    epochs=6,
    batch_size=60,
    validation_split=0.2,
)

Epoch 1/6
932/932 ━━━━━━━━━━━━━━━━━━━━ 318s 342ms/step - accuracy: 0.6054 - loss: 2.0131 - val_accuracy: 0.5549 - val_loss: 2.7280
Epoch 2/6
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.6117 - loss: 1.9551 - val_accuracy: 0.5567 - val_loss: 2.7315
Epoch 3/6
932/932 ━━━━━━━━━━━━━━━━━━━━ 317s 340ms/step - accuracy: 0.6198 - loss: 1.8905 - val_accuracy: 0.5576 - val_loss: 2.7357
Epoch 4/6
932/932 ━━━━━━━━━━━━━━━━━━━━ 317s 341ms/step - accuracy: 0.6258 - loss: 1.8401 - val_accuracy: 0.5568 - val_loss: 2.7503
Epoch 5/6
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.6301 - loss: 1.7991 - val_accuracy: 0.5580 - val_loss: 2.7565
Epoch 6/6
932/932 ━━━━━━━━━━━━━━━━━━━━ 316s 339ms/step - accuracy: 0.6355 - loss: 1.7537 - val_accuracy: 0.5579 - val_loss: 2.7620


In [46]:
## INFERENCE MODELS

encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
encoder_output_input = Input(shape=(max_len, latent_dim))

decoder_single_input = Input(shape=(1,))
dec_emb2 = Embedding(vocab_hin, latent_dim)(decoder_single_input)

decoder_outputs2, h2, c2 = decoder_lstm(
    dec_emb2,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)

attention_out2 = attention([decoder_outputs2, encoder_output_input])
concat2 = Dense(latent_dim, activation="tanh")(attention_out2)

decoder_outputs2 = decoder_dense(concat2)

decoder_model = Model(
    [decoder_single_input, encoder_output_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2, h2, c2]
)

reverse_hin = {i: w for w, i in hin_tokenizer.word_index.items()}

In [47]:
## TRANSLATE FUNCTION

def translate(sentence):
    sentence = sentence.lower()

    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    enc_out, h, c = encoder_model.predict(seq, verbose=0)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = hin_tokenizer.word_index['<start>']

    output = ""

    for _ in range(max_len):
        preds, h, c = decoder_model.predict(
            [target_seq, enc_out, h, c], verbose=0
        )

        index = np.argmax(preds[0, -1, :])
        word = reverse_hin.get(index, "")

        if word == '<end>' or word == "":
            break

        output += " " + word
        target_seq[0,0] = index

    return output.strip()


In [ ]:
## Test

print("hello →", translate("hello"))
print("how are you →", translate("how are you"))
print("i am happy →", translate("i am happy"))